<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature vector built from March 2026 data, aggregated per content item, with one engineered ratio and one categorical flag.

In [4]:
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('HF token: ')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

import pandas as pd
import numpy as np

df = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet")

agg = df.groupby('content_hash_id').agg(
    gsc_impressions_sum=('gsc_impressions', 'sum'),
    gsc_clicks_sum=('gsc_clicks', 'sum'),
    gsc_avg_position_mean=('gsc_avg_position', 'mean'),
    ga4_engaged_sessions_sum=('ga4_engaged_sessions', 'sum'),
    ga4_pageviews_sum=('ga4_pageviews', 'sum'),
    scroll_events_sum=('scroll_events', 'sum'),
    gsc_data_available=('gsc_data_available', 'max'),
).reset_index()

agg['ctr'] = (agg['gsc_clicks_sum'] / agg['gsc_impressions_sum'].replace(0, np.nan)).fillna(0)
agg['engagement_rate'] = (agg['ga4_engaged_sessions_sum'] / agg['ga4_pageviews_sum'].replace(0, np.nan)).fillna(0)
agg['gsc_available_flag'] = agg['gsc_data_available'].map({True: 1, False: 0}).fillna(0).astype(int)

feature_cols = ['gsc_impressions_sum', 'gsc_clicks_sum', 'gsc_avg_position_mean',
                'ctr', 'engagement_rate', 'scroll_events_sum', 'gsc_available_flag']
agg[feature_cols] = agg[feature_cols].fillna(0)

feature_vector = agg[['content_hash_id'] + feature_cols]
print(feature_vector.shape)
feature_vector.head()


HF token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


(331437, 8)


,content_hash_id,gsc_impressions_sum,gsc_clicks_sum,gsc_avg_position_mean,ctr,engagement_rate,scroll_events_sum,gsc_available_flag
0,content_000005d4ced12088,86,0,72.854861,0.0,0.0,0.0,1
1,content_00001e488b74b799,0,0,0.000000,0.0,0.0,0.0,0
2,content_00007bd2985b77c3,47,0,5.269565,0.0,0.0,0.0,1
3,content_00008950670cb6b5,0,0,0.000000,0.0,0.5,1.0,0
4,content_0000a348850eb1fc,0,0,0.000000,0.0,0.0,1.0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- **gsc_impressions_sum** — total impressions. Missing = 0. Available same-day.
- **gsc_clicks_sum** — total clicks. Missing = 0. Available same-day.
- **gsc_avg_position_mean** — average rank position. Missing = 0. Available same-day.
- **ctr** (engineered) — clicks/impressions. 0 if no impressions. Available same-day.
- **engagement_rate** (engineered) — engaged sessions/pageviews. 0 if no data. Available same-day.
- **scroll_events_sum** — total scrolls. Missing = 0. Available same-day.
- **gsc_available_flag** (categorical, 0/1) — whether GSC data exists. No missing. Available same-day.

All features use only same-day/same-month data — none depend on the future.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Adding a label-derived column and a future-window column on purpose, showing the score jump, then removing both.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

feature_vector = feature_vector.copy()
feature_vector['refresh_urgency'] = 1 - feature_vector['engagement_rate']

honest_cols = ['gsc_impressions_sum', 'gsc_clicks_sum', 'gsc_avg_position_mean',
               'ctr', 'scroll_events_sum', 'gsc_available_flag']
X = feature_vector[honest_cols]
y = feature_vector['refresh_urgency']

model = LinearRegression().fit(X, y)
preds = model.predict(X)
print("Honest MAE:", mean_absolute_error(y, preds))

feature_vector['leaky_label_col'] = feature_vector['refresh_urgency'] * 0.95 + np.random.normal(0, 0.01, len(feature_vector))
X1 = feature_vector[honest_cols + ['leaky_label_col']]
m1 = LinearRegression().fit(X1, y)
print("MAE with label leak:", mean_absolute_error(y, m1.predict(X1)))

feature_vector['future_engagement'] = feature_vector['engagement_rate'] + np.random.normal(0, 0.02, len(feature_vector))
X2 = feature_vector[honest_cols + ['future_engagement']]
m2 = LinearRegression().fit(X2, y)
print("MAE with future leak:", mean_absolute_error(y, m2.predict(X2)))

print("Final honest MAE (kept):", mean_absolute_error(y, preds))

Honest MAE: 0.011135671617666359
MAE with label leak: 0.008192843641384891
MAE with future leak: 0.014419516227441854
Final honest MAE (kept): 0.011135671617666359


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **client_hash_id** — retained only as an identifier, never used as a model feature. Using it directly risks learning client-specific bias rather than generalizable content signals, and raises privacy concerns.
- **ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other** — excluded due to heavy missingness (mostly NaN in this slice); including them would add noise without contributing reliable signal.
- **report_date** — used only to define the aggregation window, not included as a raw model input, since an unencoded date carries no direct predictive meaning.
- **ga4_data_available** — excluded as redundant, since it overlaps substantially with `gsc_available_flag` and adds no distinct information for this lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.